# Mini-GPT: Transformer From Scratch

## 1. Data Loading & Tokenization

In [3]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars=sorted(list(set(text)))
print(len(chars))
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
encode=lambda s:[stoi[j] for j in s]
decode=lambda l: ''.join([itos[i] for i in l])

65


## 2. Dataset & Batching

In [4]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [5]:
vocab_size=len(stoi)
block_size = 8
batch_size=4

def get_batch(data):
    starts = torch.randint(0, len(data) - block_size, (batch_size,))

    x = torch.stack( [data[start:start + block_size] for start in starts] )
    y = torch.stack( [data[start + 1:start + block_size + 1] for start in starts] )
    
    return x, y

## 3. Model Parameters

In [6]:
import torch.nn.functional as F

embedding_dim = 32
num_heads = 4
head_dim = embedding_dim // num_heads

token_embedding = (torch.randn(vocab_size, embedding_dim) * 0.02).requires_grad_()
position_embedding = (torch.randn(block_size, embedding_dim) * 0.02).requires_grad_()
# Parameters are initialized with fan-in scaling
# to keep activations and attention scores in a stable range.

Wq = (torch.randn(embedding_dim, embedding_dim) / (embedding_dim ** 0.5)).requires_grad_()
Wk = (torch.randn(embedding_dim, embedding_dim) / (embedding_dim ** 0.5)).requires_grad_()
Wv = (torch.randn(embedding_dim, embedding_dim) / (embedding_dim ** 0.5)).requires_grad_()
Wo = (torch.randn(embedding_dim, embedding_dim) / (embedding_dim ** 0.5)).requires_grad_()

W1 = (torch.randn(embedding_dim, 4 * embedding_dim) / (embedding_dim ** 0.5)).requires_grad_()
b1 = torch.zeros(4 * embedding_dim, requires_grad=True)

W2 = (torch.randn(4 * embedding_dim, embedding_dim) / ((4*embedding_dim) ** 0.5)).requires_grad_()
b2 = torch.zeros(embedding_dim, requires_grad=True)

W_out = (torch.randn(embedding_dim, vocab_size) / (embedding_dim ** 0.5)).requires_grad_()
b_out = torch.zeros(vocab_size, requires_grad=True)

gamma1 = torch.ones(embedding_dim, requires_grad=True)
beta1 = torch.zeros(embedding_dim, requires_grad=True)

gamma2 = torch.ones(embedding_dim, requires_grad=True)
beta2 = torch.zeros(embedding_dim, requires_grad=True)

In [7]:
params = [
    token_embedding,
    position_embedding,
    Wq, Wk, Wv, Wo,
    W1, b1, W2, b2,
    gamma1, beta1, gamma2, beta2,
    W_out, b_out
]

optimizer = torch.optim.Adam(params, lr=0.001)

## 4. Validation Loss

In [8]:
@torch.no_grad()
def estimate_val_loss(num_batches=40):
    losses = []
    for _ in range(num_batches):
        x, y = get_batch(val_data)
        x_emb = token_embedding[x]
        positions = torch.arange(block_size)
        x_emb = x_emb + position_embedding[positions]
        B, T, C = x_emb.shape

        mean = x_emb.mean(dim=-1, keepdim=True)
        var = x_emb.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x_emb - mean) / torch.sqrt(var + 1e-5)
        x_norm = gamma1 * x_norm + beta1

        Q = (x_norm @ Wq).view(B, T, num_heads, head_dim).transpose(1, 2)
        K = (x_norm @ Wk).view(B, T, num_heads, head_dim).transpose(1, 2)
        V = (x_norm @ Wv).view(B, T, num_heads, head_dim).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / (head_dim ** 0.5)
        mask = torch.tril(torch.ones(T, T))
        scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        attn_out = (weights @ V).transpose(1, 2).contiguous().view(B, T, embedding_dim)
        attn_out = attn_out @ Wo

        x = x_emb + attn_out
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + 1e-5)
        x_norm = gamma2 * x_norm + beta2

        ffn_out = torch.relu(x_norm @ W1 + b1) @ W2 + b2
        out = x + ffn_out
        logits = out @ W_out + b_out

        loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
        losses.append(loss.item())
    return sum(losses) / len(losses)

## 5. Training Loop

In [9]:
for step in range(20000):

    x, y = get_batch(train_data)

    x_emb = token_embedding[x]

    positions = torch.arange(block_size)
    x_emb = x_emb + position_embedding[positions]

    B, T, C = x_emb.shape

    # LayerNorm 1
    mean = x_emb.mean(dim=-1, keepdim=True)
    var = x_emb.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x_emb - mean) / torch.sqrt(var + 1e-5)
    x_norm = gamma1 * x_norm + beta1

    # Multi-Head Self-Attention
    Q = x_norm @ Wq
    K = x_norm @ Wk
    V = x_norm @ Wv

    Q = Q.view(B, T, num_heads, head_dim).transpose(1, 2)
    K = K.view(B, T, num_heads, head_dim).transpose(1, 2)
    V = V.view(B, T, num_heads, head_dim).transpose(1, 2)

    scores = Q @ K.transpose(-2, -1)
    scores = scores / (head_dim ** 0.5)

    mask = torch.tril(torch.ones(T, T))
    scores = scores.masked_fill(mask == 0, float('-inf'))

    weights = torch.softmax(scores, dim=-1)

    attn_out = weights @ V

    attn_out = attn_out.transpose(1, 2).contiguous()
    attn_out = attn_out.view(B, T, embedding_dim)

    attn_out = attn_out @ Wo

    x = x_emb + attn_out

    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x - mean) / torch.sqrt(var + 1e-5)
    x_norm = gamma2 * x_norm + beta2

    # Feed-Forward Network (FFN) 
    ffn_out = x_norm @ W1 + b1
    ffn_out = torch.relu(ffn_out)
    ffn_out = ffn_out @ W2 + b2

    out = x + ffn_out

    logits = out @ W_out + b_out

    loss = F.cross_entropy(
        logits.view(-1, vocab_size),
        y.view(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 2000 == 0:
        val_loss = estimate_val_loss()
        print(step, loss.item(), "val:", val_loss)

0 4.491935729980469 val: 4.6666698336601256
2000 2.387317419052124 val: 2.4068162113428118
4000 2.8763439655303955 val: 2.320009472966194
6000 2.678034782409668 val: 2.245617228746414
8000 2.294369697570801 val: 2.2367260217666627
10000 2.318558692932129 val: 2.2360073626041412
12000 2.0582714080810547 val: 2.1025106430053713
14000 2.3410019874572754 val: 2.2501581609249115
16000 2.358563184738159 val: 2.2886883527040482
18000 2.1127004623413086 val: 2.197061765193939


## 6. Text Generation

In [13]:
@torch.no_grad()
def generate(start_text, max_new_tokens=200):

    context = torch.tensor(encode(start_text), dtype=torch.long)

    for _ in range(max_new_tokens):

        context_crop = context[-block_size:]
        x = context_crop.unsqueeze(0)

        x_emb = token_embedding[x]

        positions = torch.arange(x.shape[1])
        x_emb = x_emb + position_embedding[positions]

        B, T, C = x_emb.shape

        mean = x_emb.mean(dim=-1, keepdim=True)
        var = x_emb.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x_emb - mean) / torch.sqrt(var + 1e-5)
        x_norm = gamma1 * x_norm + beta1

        Q = x_norm @ Wq
        K = x_norm @ Wk
        V = x_norm @ Wv

        Q = Q.view(B, T, num_heads, head_dim).transpose(1, 2)
        K = K.view(B, T, num_heads, head_dim).transpose(1, 2)
        V = V.view(B, T, num_heads, head_dim).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1)
        scores = scores / (head_dim ** 0.5)

        mask = torch.tril(torch.ones(T, T))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = torch.softmax(scores, dim=-1)

        attn_out = weights @ V
        attn_out = attn_out.transpose(1, 2).contiguous()
        attn_out = attn_out.view(B, T, embedding_dim)

        attn_out = attn_out @ Wo

        x = x_emb + attn_out

        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + 1e-5)
        x_norm = gamma2 * x_norm + beta2

        ffn_out = x_norm @ W1 + b1
        ffn_out = torch.relu(ffn_out)
        ffn_out = ffn_out @ W2 + b2

        out = x + ffn_out

        logits = out @ W_out + b_out

        logits = logits[:, -1, :]

        probs = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        context = torch.cat([context, next_token.squeeze(0)])

    return decode(context.tolist())

print(generate("SHAYNE: ", 300))

SHAYNE: Ell thoou to king upt nown.
What linh of avid;
do sher: beI prot blet him, as you epto caione ifs bas ise I not
MEIS:
Nep shou she nepearlifer;
The to to will to, the plaseerstre,
O shapperot not Es snothip nothere is to tim, bubirlonce hase. Coneve you kith louR, you far
Your this fate mus, a crot 
